**Diplay the Region Table. The columns are separated with " "**

In [0]:
path = "/Volumes/lakehouse/adventureworks_multisales/raw_zone/Product.csv"
product_uncleaned = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", "\t")
    .load(path))
display(product_uncleaned)

**Checking the data types.**
-  We need to change the Standard Cost from string to int

In [0]:
product_uncleaned.printSchema()

**Checking if we have nulls and duplicates**
- We have 0 nulls and 7 duplicates which we have to drop

In [0]:
from pyspark.sql import functions as F

count_nulls = product_uncleaned.select([F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls")
    for c in product_uncleaned.columns])
display(count_nulls)


In [0]:
total_rows = product_uncleaned.count()
unique_rows = product_uncleaned.distinct().count()

duplicate_count = total_rows - unique_rows
print("Total duplicated rows:", duplicate_count)

Inspection of inconsistent formating 
- Color : We have to change Multi and MULTI , Red and RED, Yellow and YELLOW 
- Subcategory : Wheels and Whels 
- Category : ok
- Backround Color Format :  ok
- Font Color Format: ok

In [0]:
product_uncleaned.select("Color").distinct().show()
product_uncleaned.select("Subcategory").distinct().show()
product_uncleaned.select("Category").distinct().show()
product_uncleaned.select("Background Color Format").distinct().show()
product_uncleaned.select("Font Color Format").distinct().show()



- **(Inspection of whitespaces in our columns) 
rows where the value in column changes after trimming whitespace.**

In [0]:
product_uncleaned.filter(F.col("Product") != F.trim(F.col("Product"))).show()
product_uncleaned.filter(F.col("Standard Cost") != F.trim(F.col("Standard Cost"))).show()
product_uncleaned.filter(F.col("Color") != F.trim(F.col("Color"))).show()
product_uncleaned.filter(F.col("Subcategory") != F.trim(F.col("Subcategory"))).show()
product_uncleaned.filter(F.col("Category") != F.trim(F.col("Category"))).show()
product_uncleaned.filter(F.col("Background Color Format") != F.trim(F.col("Background Color Format"))).show()
product_uncleaned.filter(F.col("Font Color Format") != F.trim(F.col("Font Color Format"))).show()




**Inspecting the Standard Cost column for empty strings, commas, symbols**
- There are no empty strings
- There are no commas
- **So the only issue is the $.**

In [0]:
from pyspark.sql import functions as F

product_uncleaned.filter(F.col("Standard Cost") == "").count()
product_uncleaned.filter(~F.col("Standard Cost").rlike("^[0-9.]+$")).show(truncate=False)
